# Attention, Deeper

Explore multi-head attention, cross-attention mechanisms, and KV caching for efficient inference. Understand how transformers achieve both representational power and computational efficiency.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/attention-deeper)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Multi-Head Attention


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention with proper dimension handling.

    Args:
        embedding_dim: Model dimension (must be divisible by num_heads)
        num_heads: Number of parallel attention heads
        dropout: Dropout probability
    """
    def __init__(self, embedding_dim, num_heads=8, dropout=0.1):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        # Single projections that output h * d_k dimensions
        self.W_q = nn.Linear(embedding_dim, embedding_dim)
        self.W_k = nn.Linear(embedding_dim, embedding_dim)
        self.W_v = nn.Linear(embedding_dim, embedding_dim)

        # Output projection
        self.W_o = nn.Linear(embedding_dim, embedding_dim)

        self.dropout = nn.Dropout(dropout)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, embedding_dim)
            mask: (seq_len, seq_len) causal mask or None

        Returns:
            output: (batch, seq_len, embedding_dim)
        """
        batch_size, seq_len, _ = x.shape

        # Project to Q, K, V: each (batch, seq_len, embedding_dim)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape for multi-head: (batch, seq_len, num_heads, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Transpose to: (batch, num_heads, seq_len, head_dim)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Compute attention: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        # Apply mask
        if mask is not None:
            # Expand mask: (seq_len, seq_len) → (1, 1, seq_len, seq_len)
            mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Softmax
        weights = F.softmax(scores, dim=-1)
        weights = torch.nan_to_num(weights, 0.0)
        weights = self.dropout(weights)

        # Aggregate: (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, head_dim)
        # = (batch, num_heads, seq_len, head_dim)
        head_outputs = torch.matmul(weights, V)

        # Reshape back: (batch, seq_len, num_heads, head_dim)
        head_outputs = head_outputs.transpose(1, 2).contiguous()

        # Concatenate heads: (batch, seq_len, embedding_dim)
        concatenated = head_outputs.view(batch_size, seq_len, self.embedding_dim)

        # Output projection
        output = self.W_o(concatenated)

        return output

# ===== Test Multi-Head Attention =====

embedding_dim = 512
num_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(embedding_dim, num_heads)
x = torch.randn(batch_size, seq_len, embedding_dim)

# Without mask
output = mha(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == x.shape

# With causal mask
mask = torch.tril(torch.ones(seq_len, seq_len))
output_causal = mha(x, mask=mask)
print(f"\n✓ Multi-head attention working correctly")
print(f"Number of heads: {num_heads}")
print(f"Dimension per head: {embedding_dim // num_heads}")
print(f"Total parameters: {sum(p.numel() for p in mha.parameters())}")

# Verify gradient flow
loss = output.sum()
loss.backward()
print(f"✓ Gradients flowing correctly")

# ===== Analyze Individual Head Attention Patterns =====

class MultiHeadAttentionDebug(nn.Module):
    """MHA with ability to return individual head attention weights"""
    def __init__(self, embedding_dim, num_heads=8):
        super().__init__()
        assert embedding_dim % num_heads == 0
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.W_q = nn.Linear(embedding_dim, embedding_dim)
        self.W_k = nn.Linear(embedding_dim, embedding_dim)
        self.W_v = nn.Linear(embedding_dim, embedding_dim)
        self.W_o = nn.Linear(embedding_dim, embedding_dim)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x, return_weights=False):
        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        weights = F.softmax(scores, dim=-1)

        head_outputs = torch.matmul(weights, V)
        head_outputs = head_outputs.transpose(1, 2).contiguous()
        concatenated = head_outputs.view(batch_size, seq_len, self.embedding_dim)
        output = self.W_o(concatenated)

        if return_weights:
            return output, weights
        return output

# Test per-head patterns
mha_debug = MultiHeadAttentionDebug(embedding_dim, num_heads)
x_test = torch.randn(1, seq_len, embedding_dim)
output_debug, head_weights = mha_debug(x_test, return_weights=True)

print(f"\n=== Head Attention Patterns ===")
print(f"Head weights shape: {head_weights.shape}")  # (batch, num_heads, seq_len, seq_len)

for head_idx in range(min(3, num_heads)):
    print(f"\nHead {head_idx} - focus pattern:")
    # Show what position 5 attends to
    focus = head_weights[0, head_idx, 5, :].detach()
    top_positions = torch.argsort(focus, descending=True)[:3]
    print(f"  Position 5 focuses on: {top_positions.tolist()}")
        

### Lesson example: Cross-Attention & KV Cache


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple

# ===== Cross-Attention =====

class CrossAttention(nn.Module):
    """Attend to different sequence (encoder-decoder, vision-language)"""
    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Q from one sequence
        self.W_q = nn.Linear(embed_dim, embed_dim)
        # K, V from different sequence
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x_query, x_key_value):
        """
        Args:
            x_query: (batch, query_len, embed_dim)
            x_key_value: (batch, kv_len, embed_dim) - different length!

        Returns:
            output: (batch, query_len, embed_dim)
        """
        batch_size = x_query.shape[0]
        query_len = x_query.shape[1]
        kv_len = x_key_value.shape[1]

        # Project
        Q = self.W_q(x_query)  # (batch, query_len, embed_dim)
        K = self.W_k(x_key_value)  # (batch, kv_len, embed_dim)
        V = self.W_v(x_key_value)

        # Reshape for heads
        Q = Q.view(batch_size, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        # (batch, num_heads, query_len, head_dim)

        K = K.view(batch_size, kv_len, self.num_heads, self.head_dim).transpose(1, 2)
        # (batch, num_heads, kv_len, head_dim)

        V = V.view(batch_size, kv_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention: different query and KV lengths
        # (batch, num_heads, query_len, head_dim) @ (batch, num_heads, head_dim, kv_len)
        # = (batch, num_heads, query_len, kv_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        weights = F.softmax(scores, dim=-1)

        # (batch, num_heads, query_len, kv_len) @ (batch, num_heads, kv_len, head_dim)
        # = (batch, num_heads, query_len, head_dim)
        head_outputs = torch.matmul(weights, V)

        # Reshape back
        head_outputs = head_outputs.transpose(1, 2).contiguous()
        concatenated = head_outputs.view(batch_size, query_len, self.embed_dim)
        output = self.W_o(concatenated)

        return output

# Test cross-attention
embed_dim = 256
batch_size = 2
encoder_len = 10  # "Hello world from source"
decoder_len = 8   # "Bonjour monde..." (being generated)

encoder_output = torch.randn(batch_size, encoder_len, embed_dim)
decoder_input = torch.randn(batch_size, decoder_len, embed_dim)

cross_attn = CrossAttention(embed_dim, num_heads=4)
output = cross_attn(decoder_input, encoder_output)

print(f"Encoder output: {encoder_output.shape}")
print(f"Decoder input: {decoder_input.shape}")
print(f"Cross-attention output: {output.shape}")
assert output.shape == decoder_input.shape
print("✓ Cross-attention works with different sequence lengths\n")

# ===== KV Cache for Inference =====

class AttentionWithKVCache(nn.Module):
    """Self-attention with KV caching for efficient decoding"""
    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(
        self,
        x: torch.Tensor,
        k_cache: Optional[torch.Tensor] = None,
        v_cache: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch, seq_len, embed_dim)
               During training: seq_len = full length
               During inference: seq_len = 1 (just new token)
            k_cache: Cached keys from previous positions or None
            v_cache: Cached values or None

        Returns:
            output: (batch, seq_len, embed_dim)
            k_cache: Updated cache
            v_cache: Updated cache
        """
        batch_size, seq_len, _ = x.shape

        # Project
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Update cache
        if k_cache is None:
            # First step: initialize cache with new K, V
            k_cache = K
            v_cache = V
        else:
            # Append new K, V to cache
            # k_cache: (batch, num_heads, cache_len, head_dim)
            # K: (batch, num_heads, 1, head_dim)
            k_cache = torch.cat([k_cache, K], dim=2)
            v_cache = torch.cat([v_cache, V], dim=2)

        # Compute attention using cached K, V
        # Q: (batch, num_heads, seq_len, head_dim) - usually 1 during inference
        # k_cache: (batch, num_heads, cache_len, head_dim)
        scores = torch.matmul(Q, k_cache.transpose(-2, -1)) * self.scale

        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, v_cache)

        # Reshape to (batch, seq_len, embed_dim)
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, self.embed_dim)
        output = self.W_o(output)

        return output, k_cache, v_cache

# Demonstrate KV cache efficiency
print("=== KV Cache Demonstration ===")

embed_dim = 512
num_heads = 8
batch_size = 1
max_seq = 100

attn_cached = AttentionWithKVCache(embed_dim, num_heads)

# Simulate inference: generate tokens one at a time
k_cache = None
v_cache = None
times = []

for step in range(10):  # Generate 10 tokens
    # Input is just 1 new token
    x_new = torch.randn(batch_size, 1, embed_dim)

    # Forward with cache
    output, k_cache, v_cache = attn_cached(x_new, k_cache, v_cache)

    # Cache size grows
    cache_size = k_cache.shape[2]
    print(f"Step {step}: Cache size = {cache_size}, Output shape = {output.shape}")

print(f"\n✓ Cache grows from 1 to {cache_size} positions")
print(f"  Without cache: recomputing keys/values at each step")
print(f"  With cache: only computing new position, then appending")
        

---

## Exercise: Build Multi-Head Attention and Implement KV Cache


Implement multi-head attention from scratch, handling the reshape operations correctly. Then add KV caching for inference. Your implementation must: (1) correctly reshape inputs to separate heads, (2) compute all heads in parallel via batched matrix ops, (3) concatenate heads after attention, (4) implement KV cache that grows during generation, (5) pass all validation tests.


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple

class MultiHeadAttentionFromScratch(nn.Module):
    """
    Multi-head attention - handle reshaping carefully!

    Args:
        embedding_dim: Model dimension (must be divisible by num_heads)
        num_heads: Number of parallel attention heads
    """
    def __init__(self, embedding_dim, num_heads=8):
        super().__init__()
        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        # TODO: Create Q, K, V projections (each embedding_dim → embedding_dim)
        # TODO: Create output projection (embedding_dim → embedding_dim)
        # TODO: Store scaling factor

    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, embedding_dim)
            mask: (seq_len, seq_len) or None

        Returns:
            output: (batch, seq_len, embedding_dim)
        """
        batch_size, seq_len, _ = x.shape

        # TODO: Project to Q, K, V (batch, seq_len, embedding_dim)

        # TODO: Reshape and transpose for heads:
        #   (batch, seq_len, embedding_dim) →
        #   (batch, seq_len, num_heads, head_dim) →
        #   (batch, num_heads, seq_len, head_dim)
        # Hint: use .view() and .transpose()

        # TODO: Compute attention scores: Q @ K^T / √head_dim
        # Result: (batch, num_heads, seq_len, seq_len)

        # TODO: Apply mask if provided (expand to (1, 1, seq_len, seq_len))

        # TODO: Apply softmax and handle NaN

        # TODO: Multiply by V: (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, head_dim)
        # Result: (batch, num_heads, seq_len, head_dim)

        # TODO: Reshape back to (batch, seq_len, embedding_dim) and apply W_o

        return output


class AttentionWithKVCacheFromScratch(nn.Module):
    """Multi-head attention with KV cache for efficient decoding"""
    def __init__(self, embedding_dim, num_heads=8):
        super().__init__()
        # Similar to above, but must handle cache in forward pass
        pass

    def forward(
        self,
        x: torch.Tensor,
        k_cache: Optional[torch.Tensor] = None,
        v_cache: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch, seq_len, embedding_dim)
            k_cache: Previous K cache or None
            v_cache: Previous V cache or None

        Returns:
            output: (batch, seq_len, embedding_dim)
            k_cache: Updated K cache
            v_cache: Updated V cache
        """
        # TODO: Compute Q, K, V as before
        # TODO: Reshape K, V to (batch, num_heads, seq_len, head_dim)
        # TODO: If cache is None, initialize with current K, V
        # TODO: If cache exists, concatenate current K, V to it (along seq dimension)
        # TODO: Compute attention using cached K, V
        # TODO: Reshape and apply output projection

        return output, k_cache, v_cache


# ===== Test Code =====
if __name__ == "__main__":
    embedding_dim = 512
    num_heads = 8
    seq_len = 10
    batch_size = 2

    # Test 1: Multi-head attention
    mha = MultiHeadAttentionFromScratch(embedding_dim, num_heads)
    x = torch.randn(batch_size, seq_len, embedding_dim)
    output = mha(x)

    assert output.shape == (batch_size, seq_len, embedding_dim)

    # Test 2: With causal mask
    mask = torch.tril(torch.ones(seq_len, seq_len))
    output_masked = mha(x, mask=mask)
    assert output_masked.shape == (batch_size, seq_len, embedding_dim)

    # Test 3: KV cache
    attn_cache = AttentionWithKVCacheFromScratch(embedding_dim, num_heads)

    # Simulate inference: 5 generation steps
    k_cache = None
    v_cache = None
    for step in range(5):
        x_new = torch.randn(batch_size, 1, embedding_dim)
        output, k_cache, v_cache = attn_cache(x_new, k_cache, v_cache)

        assert output.shape == (batch_size, 1, embedding_dim)
        assert k_cache.shape == (batch_size, num_heads, step + 1, embedding_dim // num_heads)
        assert v_cache.shape == (batch_size, num_heads, step + 1, embedding_dim // num_heads)

    print("✓ All tests passed!")
      

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/attention-deeper) for the solution and discussion.
